# Chapter 06: Vector Space Tracking & Multi-Object Association

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/06_vector_space_tracking_kalman.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How do we prove detections across video frames belong to the same vehicle and calculate smooth velocity?*

---

## 1. 🚨 The Real-World Dilemma
Sensor noise causes finite-difference velocity calculations to spike violently at 27 mph. A Kalman filter fuses physics predictions with measurement uncertainty, using Mahalanobis distance for robust data association.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

class KalmanFilter2D:
    def __init__(self, dt=0.05):
        self.F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=float)
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=float)
        self.P = np.eye(4) * 10.0
        self.Q = np.eye(4) * 0.1
        self.R = np.eye(2) * 0.5
        self.x = np.zeros((4, 1))

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x

    def update(self, z):
        y = z.reshape(2, 1) - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        return self.x

kf = KalmanFilter2D(dt=0.1)
true_pos, noisy_meas, filtered_pos = [], [], []

for t in np.arange(0, 4, 0.1):
    tx = 18.0 * t
    ty = 1.5
    true_pos.append([tx, ty])
    mx = tx + np.random.normal(0, 1.0)
    my = ty + np.random.normal(0, 0.3)
    noisy_meas.append([mx, my])
    kf.predict()
    state = kf.update(np.array([mx, my]))
    filtered_pos.append([state[0, 0], state[1, 0]])

true_pos = np.array(true_pos)
noisy_meas = np.array(noisy_meas)
filtered_pos = np.array(filtered_pos)

plt.figure(figsize=(10, 3))
plt.scatter(noisy_meas[:, 0], noisy_meas[:, 1], color="#f85149", alpha=0.5, label="Raw Noisy Detections")
plt.plot(true_pos[:, 0], true_pos[:, 1], 'k--', label="True Motion")
plt.plot(filtered_pos[:, 0], filtered_pos[:, 1], color="#3fb950", lw=2, label="Kalman Filter")
plt.title("Kalman Filter Noise Rejection in Vector Space")
plt.xlabel("Forward X (m)")
plt.ylabel("Lateral Y (m)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Track ID switches when cars cross** | Euclidean association ambiguity in close proximity. | Compute cross-track Mahalanobis distance. | Use Hungarian Algorithm with appearance embeddings (Re-ID). |
| **Covariance matrix becomes non-positive definite** | Numerical roundoff in $(I - KH)P$. | Check eigenvalues $\lambda_i \le 0$. | Use Joseph Form: $P = (I - KH)P(I - KH)^T + KRK^T$. |